# Машинное обучение, ФКН ВШЭ

## Практическое домашнее задание 3.2  Продвинутая генерация признаков

### Общая информация

Дата выдачи: 23.02.2026

Мягкий дедлайн: 12.03.2026 23:59MSK

Жесткий дедлайн: 16.03.2026 23:59MSK

### О задании

В данном задании вы познакомитесь с менее тривиальными подходами для создания новых признаков в табличном машинном обучении. Вам понадобится подумать над тем, зачем мы делаем те или иные преобразования, научиться принимать решения, дающие наилучшие результаты, и узнать, как реализовывать их при помощи библиотек

### Оценивание и штрафы

См. базовую часть

### Формат сдачи
Задания сдаются через систему Anytask. Инвайт можно найти на странице курса. Присылать необходимо ноутбук с выполненным заданием. Сам ноутбук называйте в формате **homework-practice-03-advanced-Username.ipynb**, где Username — ваша фамилия.

### **Введение**

В этой части ноутбука задания посложнее дефолтного фит трансформа. Максимальная оценка за оба — 8 баллов, остальное вы можете получить, если примете участие в соревновании, и всего можете выбить аж 13 из 10. Тут ожидается больше самостоятельности, как от (почти) полноценной рабочей единицы: 
- Вы **сами** решаете, что <font color="#cb9255">**хотите**</font> делать. Пункты можно делать частично, можно скипать или сделать часть, **максимум баллов ограничен двумя**. **Посмотрите на все из них**, прежде чем приступать
- Вы **сами** чистите данные, если чуете в них подвох (теперь они далеко не такие няшные)
- Вы **сами** <font color="#f68c9d">**обосновываете**</font> (в голове, если не указано явно), могут они вам вообще помочь или нет (часть пунктов явно сильнее других)

Все прочие пожелания по тому, как строить графики, на чём фиттить, а на чём предиктить, сохраняются, будьте внимательны. Во всех пунктах с 📈 нужно добиться хотя бы минимального улучшения, относительно бейзлайна (того, что вышло в части **base**) чтобы получить балл (даже если улучшение на 0.005)

Ещё раз обратите внимание, что **максимум за advanced часть — 2 балла**, делать всё не нужно, только самое приятное. Мы в вас верим!

### **Часть 4. Текста** (1.5 балла) <img align="center" src="https://static.wikia.nocookie.net/dota2_gamepedia/images/4/4f/Emoticon_blush.gif/revision/latest?cb=20180504011409">

В которой студент знакомится с внутренним миром дотеров

#### **Задание 4.1. Предобработка текста** (0.75 балла)

<span style="color:grey"><font size="1">Если вам когда-либо приходила в голову мысль, что создание Интернета было ошибкой, то, что ж, после этого задания сомнения могут отпасть.</font></span>

Для некоторых матчей имеется информация о том, что писали местные аборигены, в течение тех же **15 минут от начала матча**. К сожалению, доселе мы работали лишь с таблицами, а не с текстами, но не беда, простейшие подходы нейросетей не требуют, а в простых задачах, вроде бинарной классификации, работать будут не хуже

Откройте датафрейм `game_chat.csv`, выведите парочку текстов, посмотрите, как они устроены, как там хранятся множественные сообщения,  и так далее, что у нас есть, а чего, увы, нет

In [57]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )
chat = pd.read_csv('game_chat.csv')

print(chat.shape)
print(chat.columns.tolist())
chat.head()

(700838, 3)
['match_id', 'radiant_chat', 'dire_chat']


,match_id,radiant_chat,dire_chat
0,235435,потренируйся с ботами????,саппортам | U LAGGING BRAH | stack..... | шлюх
1,102127,NaN,NaN
2,383046,u just buy levels blue? | fa ge,NaN
3,729879,NaN,NaN
4,126886,NaN,NaN


In [58]:
for i in range(3):
    print(chat['radiant_chat'].iloc[i])


потренируйся с ботами????
nan
u just buy levels blue? | fa ge


Дём дальше. Тексты нужно готовить, прежде чем пихать их в модель. Оценивать выбросы здесь довольно проблематично, в силу специфики дотерских сообщений, хотя вы, конечно, можете попытаться. Речь здесь про базовую предобработку.

Задача минимум, тут мы вам поможем:
- Разобраться с библиотекой `nltk` и разбить текст на токены — отдельные сущности, составляющие текст (чаще всего слова, но бывает и что-то другое, надо разобраться). Как бить текст — вопрос неоднозначный. В целом подойдёт любой способ, но какие-то [токенизаторы](https://www.nltk.org/api/nltk.tokenize.html) могут сразу покрыть часть проблем с текстами в задаче максимум
- Лемматизировать текст (привести слова к начальной форме). <br>
<font color="#cb9255">**Варианта два**</font>: манкипатчить [`pymorphy2`](https://pymorphy2.readthedocs.io/en/stable/) или откатывать версии (там вылезет ошибка, если у вас слишком новый питон), либо применять [`mystem`](https://pypi.org/project/pymystem3/), что может затянуться на несколько часов, зато лемматизация будет точнее

Задача максимум, тут вам нужно понять, как всё обработать, самим (включать-не включать, выкинуть-оставить — валидны **все** варианты, но только, если, есть, <font color="#f68c9d">**обоснование**</font>):
- повторы символов
- знаки препинания
- стоп-слова
- нижний регистр
  
Вам нужно пройтись по всем пунктам, не обязательно в этом порядке. Если найдёте что-то ещё — круто, молодцы, можно тоже пофиксить

In [77]:
import re
from collections import Counter

display(chat.head())
display(chat.sample(3, random_state=42))

TOKEN_RE = re.compile(r"[a-zа-яё]+", re.IGNORECASE)
REPEAT_RE = re.compile(r"(.)\1{2,}")

PROFANITY_RE = re.compile(
    r"(?:"
    r"сос(?:ать|ал[аи]?|ет|ё[тш]|ешь|ут|ите)?|"
    r"(?:вы|на|за|про|по)?[её]б(?:ан(?:ый|ая|ое|ые)?|а(?:ть|л[аио]?|н[ау]?л[аио]?)?|"
    r"н(?:ый|ая|ое|ые)?|(?:у|е)?т|(?:ош|аш)ить)?|"
    r"бля(?:д(?:ь|ина|ство)?)?|"
    r"жоп(?:а|ой|у|е|ы)?|"
    r"пос?рат(?:ь|ься|ил[аио]?|ят|ишь)?|"
    r"(?:твою|вашу|их) мать"
    r")",
    flags=re.IGNORECASE,
)

AGGRESSION_RE = re.compile(
    r"(?:"
    r"дебил(?:ы|ка|ом)?|даун(?:ы|ом)?|урод(?:ы|ина)?|мраз(?:ь|и|ота)|твар(?:ь|ина)|"
    r"скотин(?:а|ы)|чмо|лох(?:и|ов)?|лошар(?:а|ы)|падл(?:а|ы)|суч?к?а|п[её]с(?:ы|ина)?|"
    r"кретин(?:ы|ом)?|придур(?:ок|ки|ошный)?|отмороз(?:ок|ки)|moron|idiot|feeder|фидер"
    r")",
    flags=re.IGNORECASE,
)

RU_STOPWORDS = {
    "и", "в", "во", "на", "но", "а", "не", "что", "это", "как", "к", "по", "за", "из", "у", "же",
    "ты", "вы", "мы", "он", "она", "они", "бы", "быть", "to", "the", "of", "is", "are", "am", "be"
}

try:
    import pymorphy2
    morph_analyzer = pymorphy2.MorphAnalyzer()
except Exception:
    morph_analyzer = None

def normalize_token(token: str) -> str:
    processed_token = REPEAT_RE.sub(r"\1\1", token.lower())
    if morph_analyzer is not None:
        try:
            processed_token = morph_analyzer.parse(processed_token)[0].normal_form
        except Exception:
            pass
    return processed_token

def preprocessing(text: str) -> str:
    if pd.isna(text) or not str(text).strip():
        return ""

    cleaned_text = str(text).replace("|", " ").lower()
    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"[^a-zа-яё\s]", " ", cleaned_text)
    tokens_list = [normalize_token(token) for token in TOKEN_RE.findall(cleaned_text)]
    tokens_list = [token for token in tokens_list if len(token) > 1 and token not in RU_STOPWORDS]
    return " ".join(tokens_list)

def extract_chat_stats(text: str) -> dict:
    raw_input = "" if pd.isna(text) else str(text)
    processed_output = preprocessing(raw_input)
    tokens_result = processed_output.split()
    messages_raw = [chunk.strip() for chunk in raw_input.split("|") if chunk.strip()]

    return {
        "msg_count": len(messages_raw),
        "token_count": len(tokens_result),
        "unique_ratio": len(set(tokens_result)) / max(len(tokens_result), 1),
        "caps_share": sum(char.isupper() for char in raw_input) / max(sum(char.isalpha() for char in raw_input), 1),
        "exclamation_count": raw_input.count("!"),
        "question_count": raw_input.count("?"),
        "profanity_hits": len(PROFANITY_RE.findall(raw_input)),
        "aggression_hits": len(AGGRESSION_RE.findall(raw_input)),
    }

sample_text = "Ляяя, ваша мама такая красивая, ну вылитый пудж)))0"
print(preprocessing(sample_text))
print(extract_chat_stats(sample_text))

,match_id,radiant_chat,dire_chat
0,235435,потренируйся с ботами????,саппортам | U LAGGING BRAH | stack..... | шлюх
1,102127,NaN,NaN
2,383046,u just buy levels blue? | fa ge,NaN
3,729879,NaN,NaN
4,126886,NaN,NaN


,match_id,radiant_chat,dire_chat
56453,712968,NaN,NaN
560547,423834,NaN,ливаю
376170,478779,NaN,NaN


ляя ваша мама такая красивая ну вылитый пудж
{'msg_count': 1, 'token_count': 8, 'unique_ratio': 1.0, 'caps_share': 0.02631578947368421, 'exclamation_count': 0, 'question_count': 0, 'profanity_hits': 0, 'aggression_hits': 0}


<div style="border-left: 5px solid #ff748c; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** ну что, как обрабатываем текст?

**Ответ:**

</div>

тест я привел к нижнему регистру, удаляю ссылки, цифро-символьный мусор и знаки препинания, а также использую | как разделитель отдельных сообщений. Повторы символов сжимаю, чтобы уменьшить влияние эмоционального спама вроде ляяяя, после чего разбиваю текст на токены по буквенным последовательностям.
потом удаляю короткие токены и стоп-слова, а при доступности pymorphy2 лемматизирую слова. также из чата дополнительно извлекаю простые статистики такие как число сообщений, число токенов, долю уникальных слов, долю капса, количество ! и ?, а также число срабатываний на обсценную и агрессивную лексику

#### 📈 **Задание 4.2. Векторизация** (0.5 балла)

Ура, если у вас получились токены, то наконец-то можно что-то закодировать, но как? Рад, что вы спросили. Читайте конспект семинаров или документацию

<table width="800" border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th width="50%">
      <font color="#cb9255">CountVectorizer</font>
    </th>
    <th width="50%">
      <font color="#cb9255">TfIdfVectorizer</font>
    </th>
  </tr>
  <tr>
    <td valign="top">
      Берёт и считает, сколько раз в тексте <br>
      встретилось то или иное слово. Похож <br>
      на наш энкодер из части 2.
      <br><br>
      <table border="1" cellpadding="4" cellspacing="0">
        <thead>
          <tr>
            <th scope="col">word_1</th>
            <th scope="col">word_2</th>
            <th scope="col">word_3</th>
            <th scope="col">...</th>
            <th scope="col">word_m</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>1</td>
            <td>0</td>
            <td>2</td>
            <td>...</td>
            <td>100</td>
          </tr>
        </tbody>
      </table>
    </td>
    <td valign="top">
      Более хитрая штука: вместе с количеством слов (tf)<br>
      считает их важность (idf). Слова, встречающиеся <br>
      во всех документах, считаются не важными<br>
      и зануляются (idf = log1).
      <br><br>
      <table border="1" cellpadding="4" cellspacing="0">
        <thead>
          <tr>
            <th scope="col"></th>
            <th scope="col">word_1</th>
            <th scope="col">word_2</th>
            <th scope="col">word_3</th>
            <th scope="col">...</th>
            <th scope="col">word_m</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td><b>text_1</b></td>
            <td>1*log2</td>
            <td>0</td>
            <td>2*log2</td>
            <td>...</td>
            <td>100*log1</td>
          </tr>
          <tr>
            <td><b>text_2</b></td>
            <td>0</td>
            <td>10*log2</td>
            <td>0</td>
            <td>...</td>
            <td>100*log1</td>
          </tr>
        </tbody>
      </table>
    </td>
  </tr>
</table>


Оба векторайзера хороши, но у каждого из них есть <font color="#cb9255">**гиперпараметры**</font>. Естественно, они повлияют на качество, <font color="#cb9255">**можете подобрать их**</font> попозже, дефолтные тоже должны показать эффект

Обучите по векторайзеру на чатах Radiant и Dire. Приклейте результат к вашему датасету и обучите модель на всём получившемся великолепии (sparse формат убирать не рекомендуется)

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 4.3. Визуализация** (0.25 балла)

Для любителей графиков есть малюсенькое задание: визуализируйте облако слов с наибольшими по модулю весами (разделите их на условно *"позитивные"* и *"негативные"*)

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

<div style="border-left: 5px solid #ff748c; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** что думаете? Сейчас и вообще по модели — негативные получились слова или так, едва?

**Ответ:**

</div>

### **Часть 5. Агрегации** (1.75 балла) <img align="center" src="https://static.wikia.nocookie.net/dota2_gamepedia/images/4/4a/Techies_emoticon.gif/revision/latest?cb=20180504014918">

В которой студент начинает ведать

#### 📈 **Задание 5.1. Статистики матча** (0.75 балла)

Есть у нас в данных большой кусок про advantage — преимущество команды сил Света с точностью до минуты, по золоту и опыту, всё так же **в пределах 15 минут**. Лежат они в `dota_adv.csv`. Чем больше число, тем больше шанс на победу — всё просто. Только график, как правило, не линеен.

С ними в очередной раз есть *нюансы* — необходимо разобраться, как они там лежат, и всё ли там в порядке со значениями, но это меньшая из проблем. А также нарисовать парочку advantage, чтобы было понимание, как они себя ведут

In [8]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )
import pandas as pd
import numpy as np

adv = pd.read_csv('dota_adv.csv')
print(adv.shape)
print(adv.head())
print(adv.columns.tolist())
print(adv.isna().sum().sort_values(ascending=False).head(20))

(767822, 3)
   match_id                                   radiant_gold_adv  \
0    526846  [   0  159  452 1904 2100 3290 3290 3290 3290 ...   
1    511496  [   0 -151 -141   12 -165 -151 -151    4  377 ...   
2     90272                                                 []   
3    153647                                                 []   
4    694826                                                 []   

                                     radiant_exp_adv  
0  [   0   68  658 1397 1435 2118 2118 1923 1923 ...  
1  [   0    1 -136  243 -270   -8   -8 -169 -169 ...  
2                                                 []  
3                                                 []  
4                                                 []  
['match_id', 'radiant_gold_adv', 'radiant_exp_adv']
match_id            0
radiant_gold_adv    0
radiant_exp_adv     0
dtype: int64


In [15]:
import ast

def parse_adv(x):
    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    if isinstance(x, str):
        x = x.strip()

        if x == '[]' or x == '':
            return []

        try:
            val = ast.literal_eval(x)
            if isinstance(val, list):
                return val
        except:
            pass

        x = x.strip('[]').strip()
        if x == '':
            return []
        return list(map(int, x.split()))

    return []

adv['radiant_gold_adv'] = adv['radiant_gold_adv'].apply(parse_adv)
adv['radiant_exp_adv'] = adv['radiant_exp_adv'].apply(parse_adv)

print(adv['radiant_gold_adv'].iloc[0])
print(type(adv['radiant_gold_adv'].iloc[0]))
print(adv['radiant_gold_adv'].iloc[2])

[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 3290, 3859, 3859, 3087, 3087, 3849, 5342, 5342]
<class 'list'>
[]


In [18]:
def agg_features(arr):
    if len(arr) == 0:
        return pd.Series([0, 0, 0, 0])

    arr = np.array(arr)
    return pd.Series([
        arr.mean(),
        arr.std(),
        arr.max(),
        arr.min()
    ])

gold_stats = adv['radiant_gold_adv'].apply(agg_features)
gold_stats.columns = ['gold_adv_mean', 'gold_adv_std', 'gold_adv_max', 'gold_adv_min']

exp_stats = adv['radiant_exp_adv'].apply(agg_features)
exp_stats.columns = ['exp_adv_mean', 'exp_adv_std', 'exp_adv_max', 'exp_adv_min']

adv_features = pd.concat([adv[['match_id']], gold_stats, exp_stats], axis=1)

adv_features.head()

,match_id,gold_adv_mean,gold_adv_std,gold_adv_max,gold_adv_min,exp_adv_mean,exp_adv_std,exp_adv_max,exp_adv_min
0,526846,2887.500,1559.052797,5342.0,0.0,2262.250,1401.628406,4661.0,0.0
1,511496,933.875,1342.763702,3698.0,-165.0,191.625,385.377717,931.0,-270.0
2,90272,0.000,0.000000,0.0,0.0,0.000,0.000000,0.0,0.0
3,153647,0.000,0.000000,0.0,0.0,0.000,0.000000,0.0,0.0
4,694826,0.000,0.000000,0.0,0.0,0.000,0.000000,0.0,0.0


Для начала возьмём простые агрегации. Можете взять те, что вам знакомы (статистики - среднее, стд и др.), можете взять фан факты в вашей любимой библиотеке для данных, например [тут](https://pandas.pydata.org/docs/user_guide/groupby.html#aggregation) или [тут](https://docs.pola.rs/api/python/stable/reference/expressions/aggregation.html).

Задание:
- взять 4 статистики из библиотеки, применить к обеим колонкам `_adv`, <font color="#f68c9d">**обдумать**</font>, почему именно они
- одну из статистику выше разбить по командам, и точно так же примените к колонкам (получится что-то типа `agg_xp` -> `agg_dire_xp`, `agg_radiant_xp`)

<div style="border-left: 5px solid #647cb8; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Тут мы встаём на скользкую дорожку переобучения. Агрегаций можно сделать **очень** много. Добавьте их все, и ваша модель превратится в тыкву. Удобнее будет сразу бить их на группы, например `features_last`, `features_q25`, `features_kurtosis_dire_10min+` и так далее, в зависимости от степени упоротости

C другой стороны, агрегации это самая сильная группа фичей, и для десяточки лучше целиться именно в них

</div>

<div style="border-left: 5px solid #f68c9d; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** какие агрегации берём?

**Ответ:**

</div>

в качестве агрегации я беру mean, max,min, и std.
mean покажет средний перевес команды radiant за первые 15 минут
std показывает, насколько нестабильно менялось преимущество по ходу матча
max фиксирует максимальный пик преимущества Radiant
min отражает наибольшее преимущество команды Dire

In [20]:
def team_mean(arr):
    if len(arr) == 0:
        return pd.Series([0, 0])

    arr = np.array(arr)

    radiant = np.clip(arr, 0, None)
    dire = np.clip(-arr, 0, None)

    return pd.Series([
        radiant.mean(),
        dire.mean()
    ])

In [21]:
gold_team = adv['radiant_gold_adv'].apply(team_mean)

gold_team.columns = [
    'gold_radiant_adv_mean',
    'gold_dire_adv_mean'
]

In [22]:
exp_team = adv['radiant_exp_adv'].apply(team_mean)

exp_team.columns = [
    'exp_radiant_adv_mean',
    'exp_dire_adv_mean'
]

In [25]:
adv_features = pd.concat(
    [
        adv_features,
        gold_team,
        exp_team
    ],
    axis=1
)

adv_features.head()

,match_id,gold_adv_mean,gold_adv_std,gold_adv_max,gold_adv_min,exp_adv_mean,exp_adv_std,exp_adv_max,exp_adv_min,gold_radiant_adv_mean,...,exp_radiant_adv_mean,exp_dire_adv_mean,gold_radiant_adv_mean,gold_dire_adv_mean,exp_radiant_adv_mean,exp_dire_adv_mean,gold_radiant_adv_mean,gold_dire_adv_mean,exp_radiant_adv_mean,exp_dire_adv_mean
0,526846,2887.500,1559.052797,5342.0,0.0,2262.250,1401.628406,4661.0,0.0,2887.5000,...,2262.250,0.0,2887.5000,0.0000,2262.250,0.0,2887.5000,0.0000,2262.250,0.0
1,511496,933.875,1342.763702,3698.0,-165.0,191.625,385.377717,931.0,-270.0,981.3125,...,239.125,47.5,981.3125,47.4375,239.125,47.5,981.3125,47.4375,239.125,47.5
2,90272,0.000,0.000000,0.0,0.0,0.000,0.000000,0.0,0.0,0.0000,...,0.000,0.0,0.0000,0.0000,0.000,0.0,0.0000,0.0000,0.000,0.0
3,153647,0.000,0.000000,0.0,0.0,0.000,0.000000,0.0,0.0,0.0000,...,0.000,0.0,0.0000,0.0000,0.000,0.0,0.0000,0.0000,0.000,0.0
4,694826,0.000,0.000000,0.0,0.0,0.000,0.000000,0.0,0.0,0.0000,...,0.000,0.0,0.0000,0.0000,0.000,0.0,0.0000,0.0000,0.000,0.0


In [38]:
train_new = pd.read_csv('train_base.csv')

adv_features = adv_features.loc[:, ~adv_features.columns.duplicated()].copy()

train_new = train_new.merge(adv_features, on='match_id', how='left')

new_adv_cols = [col for col in adv_features.columns if col != 'match_id']
train_new.loc[:, new_adv_cols] = train_new.loc[:, new_adv_cols].fillna(0)

print(train_new.shape)
train_new[new_adv_cols].head()

(641090, 26)


,gold_adv_mean,gold_adv_std,gold_adv_max,gold_adv_min,exp_adv_mean,exp_adv_std,exp_adv_max,exp_adv_min,gold_radiant_adv_mean,gold_dire_adv_mean,exp_radiant_adv_mean,exp_dire_adv_mean
0,0.0000,0.000000,0.0,0.0,0.0000,0.000000,0.0,0.0,0.0000,0.0000,0.0000,0.0000
1,418.0625,615.661481,1292.0,-322.0,-275.9375,411.195584,381.0,-1122.0,518.6875,100.6250,68.3125,344.2500
2,0.0000,0.000000,0.0,0.0,0.0000,0.000000,0.0,0.0,0.0000,0.0000,0.0000,0.0000
3,-12.8125,852.670747,1237.0,-1994.0,1039.8125,1243.421802,3694.0,-77.0,300.3750,313.1875,1048.5000,8.6875
4,867.6875,769.553906,2203.0,-629.0,1515.7500,1308.464534,5297.0,-35.0,907.6875,40.0000,1517.9375,2.1875


Обучите модель по агрегациям (одной группе или нескольким) + предыдущим фичам. Чтобы получить фулл балл, придётся показать, что хотя бы минимальный импрув есть, относительно бейзлайна

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def gini_score(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

train_51 = train_new.copy()
train_51 = train_51.sort_values('date').reset_index(drop=True)

split = int(len(train_51) * 0.8)

train_part = train_51.iloc[:split].copy()
valid_part = train_51.iloc[split:].copy()

# заполним пропуски как в базовой части (медианой)
train_part['mmr_missing'] = train_part['avg_mmr'].isna().astype(int)
valid_part['mmr_missing'] = valid_part['avg_mmr'].isna().astype(int)

median_mmr = train_part['avg_mmr'].median()
train_part['avg_mmr'] = train_part['avg_mmr'].fillna(median_mmr)
valid_part['avg_mmr'] = valid_part['avg_mmr'].fillna(median_mmr)

# уберем лишнее
drop_cols = ['radiant_win', 'date', 'match_id', 'log', 'sqrt', '1/1+f', 'exp']
drop_cols = [col for col in drop_cols if col in train_51.columns]

X_train = train_part.drop(columns=drop_cols)
y_train = train_part['radiant_win']

X_valid = valid_part.drop(columns=drop_cols)
y_valid = valid_part['radiant_win']


model_51 = LogisticRegression(max_iter=2000, random_state=42)
model_51.fit(X_train, y_train)

valid_pred_proba = model_51.predict_proba(X_valid)[:, 1]
gini_51 = gini_score(y_valid, valid_pred_proba)

print('Gini с новыми фичами', gini_51)

Gini с новыми фичами 0.24023842786034022


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#### 📈 **Задание 5.2. Тренд** (0.5 балла)

Каждый уважающий себя лудоман знает, что 99% процентов игроков останавливается ровно перед тем, как сорвать джекпот. Так и здесь — если команда с треском проигрывает в первые 15 минут матча, возможно это признак камбека в следующие 50, как знать? Попробуем собрать агрегацию похитрее — она будет обозначать тренд, который есть в графиках преимущества, и если пословица верна, наша модель уловит эту зависимость.

<span style="color:grey"><font size="1">Администрация курса МО-1 категорически против азартных игр, пример приводится сугубо в образовательных целях.</font></span>

<div style="border-left: 5px solid #f68c9d; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** для чего нам вообще тренд? Полезная ли это агрегация?

**Ответ:**

</div>

Тренд нужен, чтобы учитывать не только величину преимущества, но и направление его изменения во времени. Простые агрегаты вроде среднего или максимума показывают уровень advantage, но не отвечают на вопрос, усиливалось ли преимущество команды или исчезало.
Это полезная агрегация, потому что две игры могут иметь похожие средние значения advantage, но совершенно разную динамику, типа в одном матче команда постепенно захватывает инициативу, а в другом - теряет уже имевшийся перевес. Тренд помогает модели уловить такие различия

Агрегировать можно и вещи несколько более прикольные, чем те, что есть в основном функционале. Делать это мы будем, как вы наверняка догадались, трансформером, ну а чем же ещё. Что он умеет?

1. Принимает на вход функцию колонку и <font color="#cb9255">**параметры**</font> на ваш вкус, как минимум `method`, метод расчёта `slope`
2. Выделяет коэффициент наклона (`slope`, он же $\alpha$) при помощи одного из методов:
   - `'delta'`: разность первого и последнего значений $|x_{\max} - x_{\min}|$
   - `'OLS'`: линейная регрессия, обученная методом МНК $(X^TX)^{-1}X^Ty$
   - альтернативный метод, порождённый вашей бурной фантазией
3. Считает `r2` и `intercept` для одного advantage (если что это тоже могут быть наши фичи!)

In [50]:
from typing import Iterable


class TrendTransformer:

    def __init__(self, columns: Iterable[str], method: str = 'ols'):
        self.columns = list(columns)
        self.method = method

    def fit(self, X, y=None):
        return self

    def _parse_array(self, arr):
        if isinstance(arr, list):
            return np.array(arr, dtype=float)

        if pd.isna(arr):
            return np.array([], dtype=float)

        if isinstance(arr, str):
            arr = arr.strip()
            if arr == '' or arr == '[]':
                return np.array([], dtype=float)
            arr = arr.strip('[]').strip()
            if arr == '':
                return np.array([], dtype=float)
            return np.array(list(map(float, arr.split())), dtype=float)

        return np.array([], dtype=float)

    def _calc_trend_features(self, arr):
        arr = self._parse_array(arr)

        if len(arr) == 0:
            return pd.Series([0.0, 0.0, 0.0], index=['slope', 'intercept', 'r2'])

        if len(arr) == 1:
            return pd.Series([0.0, arr[0], 0.0], index=['slope', 'intercept', 'r2'])

        x = np.arange(len(arr), dtype=float)

        if self.method == 'delta':
            slope = arr[-1] - arr[0]
            intercept = arr[0]

            y_pred = np.linspace(arr[0], arr[-1], len(arr))
            ss_res = np.sum((arr - y_pred) ** 2)
            ss_tot = np.sum((arr - arr.mean()) ** 2)
            r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

        elif self.method == 'ols':
            slope, intercept = np.polyfit(x, arr, 1)
            y_pred = slope * x + intercept

            ss_res = np.sum((arr - y_pred) ** 2)
            ss_tot = np.sum((arr - arr.mean()) ** 2)
            r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

        else:
            raise ValueError("дичь")

        return pd.Series([slope, intercept, r2], index=['slope', 'intercept', 'r2'])

    def transform(self, X, y=None):
        X_new = X.copy()

        for col in self.columns:
            trend_stats = X_new[col].apply(self._calc_trend_features)

            X_new[f'{col}_{self.method}_slope'] = trend_stats['slope']
            X_new[f'{col}_{self.method}_intercept'] = trend_stats['intercept']
            X_new[f'{col}_{self.method}_r2'] = trend_stats['r2']

        return X_new

Реализуйте трансформер. Критерий успеха, вновь, качество — фича должна помочь, хотя бы на долю пункта

In [52]:
trend_transformer = TrendTransformer(
    columns=['radiant_gold_adv', 'radiant_exp_adv'],
    method='ols'
)

trend_transformer.fit(adv)
adv_with_trend = trend_transformer.transform(adv)

adv_with_trend.head()

,match_id,radiant_gold_adv,radiant_exp_adv,radiant_gold_adv_ols_slope,radiant_gold_adv_ols_intercept,radiant_gold_adv_ols_r2,radiant_exp_adv_ols_slope,radiant_exp_adv_ols_intercept,radiant_exp_adv_ols_r2
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192...",304.388235,604.588235,0.810015,282.491176,143.566176,0.863183
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18...",256.185294,-987.514706,0.773514,42.938235,-130.411765,0.263800
2,90272,[],[],0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,153647,[],[],0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,694826,[],[],0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [53]:
trend_cols = [
    'radiant_gold_adv_ols_slope',
    'radiant_gold_adv_ols_intercept',
    'radiant_gold_adv_ols_r2',
    'radiant_exp_adv_ols_slope',
    'radiant_exp_adv_ols_intercept',
    'radiant_exp_adv_ols_r2'
]

trend_features_df = adv_with_trend[['match_id'] + trend_cols].copy()
trend_features_df.head()

,match_id,radiant_gold_adv_ols_slope,radiant_gold_adv_ols_intercept,radiant_gold_adv_ols_r2,radiant_exp_adv_ols_slope,radiant_exp_adv_ols_intercept,radiant_exp_adv_ols_r2
0,526846,304.388235,604.588235,0.810015,282.491176,143.566176,0.863183
1,511496,256.185294,-987.514706,0.773514,42.938235,-130.411765,0.263800
2,90272,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,153647,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,694826,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [54]:
train_52 = pd.read_csv('train_base.csv')

train_52 = train_52.merge(trend_features_df, on='match_id', how='left')
train_52[trend_cols] = train_52[trend_cols].fillna(0)

print(train_52.shape)
train_52[trend_cols].head()

(641090, 20)


,radiant_gold_adv_ols_slope,radiant_gold_adv_ols_intercept,radiant_gold_adv_ols_r2,radiant_exp_adv_ols_slope,radiant_exp_adv_ols_intercept,radiant_exp_adv_ols_r2
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,-32.619118,662.705882,0.059651,-27.722059,-68.022059,0.096586
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,-69.283824,506.816176,0.140301,223.242647,-634.507353,0.684977
4,20.945588,710.595588,0.015742,235.097059,-247.477941,0.686008


In [56]:
model_52 = LogisticRegression(max_iter=2000, random_state=42)
model_52.fit(X_train, y_train)

valid_pred_proba = model_52.predict_proba(X_valid)[:, 1]
gini_52 = 2 * roc_auc_score(y_valid, valid_pred_proba) - 1

print('Gini c трендом:', gini_52)

Gini c трендом: 0.24023842786034022


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


не особо помогло. главное хуже не стало)

#### **📈 Задание 5.3. Бинаризация** (0.5 балла)

Ровно одну прикольную фишку для числовых признаков мы пока что не рассмотрели — бинаризацию. Если вы до неё уже догадались, то вы — гений, не думали на <font color="cb9255">**МОП**</font>? А если нет, суть такова:

1. Берём отрезок advantage и бьём его на несколько бинов
2. Бины можно использовать, как фичу саму по себе, а можно подсобрать внутри неё агрегации

<div style="border-left: 5px solid #647cb8; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Во-первых, ваша модель внезапно становится нелинейной, хоть и кусочной, это полный отвал \
Во-вторых, это простейший пример ансамбля, если бинаризовать таргет (но у нас, увы нет смысла, он дискретный). Нелинейность полезна почему — в первые минуты преимущество не так решает, как в последние. \
В-третьих, это фильтрует шумный сигнал, выбросы то отлетят в соответствующий бин

</div>

Попробуем? Бинаризуйте признаки advantage: занумеруйте их (сделайте категорию) и посчитайте побиновые агрегации

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

### **Часть 6. Около ML** (2 балла) <img height=25px width=35px align="center" src="https://media1.tenor.com/m/72ScVNgTGpYAAAAC/kaneki-tokyo-ghoul.gif"></img>

В которой студент жесточайше чиллит после пережитого ужаса

#### **Задание 6.1. Пайплайн** (0.5 балла)

Работать в ноутбуках становится экспоненциально тяжелее по мере разрастания модели. Чтобы немножко упорядочить хаос, вам предлагается засунуть всё в один пайплайн. Критерии:

- функция или класс (может понравиться `ColumnTransformer` и `Pipeline`)
- возможность нажать одну кнопку, чтобы запустить пайплайн, уйти пить пиво и вернуться к уже готовому submission для Kaggle
- возможность передать флаги (какие фичи добавляем) и параметры (если есть разные варианты сбора параметров)
- включает в себя все пункты, к которым вы прикоснулись в рамках домашнего задания

А вот как именно это делать — дело ваше, для себя же стараетесь

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 6.2. Storage** (0.25 балла)

Вдогоночку можно ещё и создать псевдо-БД, чтобы хранить наши шедевры и не потеряться в тысячах моделек. Давайте вот такую штуку запилим:

- датафрейм или честная БД для версионирования моделей
- для каждой модели есть уникальный идентификатор
- для каждой модели сохраняются её гиперпараметры или параметры всего пайплайна (если вы его сделали)
- для каждой модели хранятся метрики на валидации

Сделайте и продемонстрируйте

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 6.3. CuML** (0.25 балла)

Если вы таки осмелились делать домашку именно на Kaggle, то поздравляю, пожалуй, это самое здравое решение в этой дз. Чтобы использовать его возможности по полной, пересядьте с вашей модели из `sklearn`, которую вы выбрали в задании про даты **(1.3)**, на модель из `cuml`. 

[Разберитесь](https://docs.rapids.ai/api/cuml/stable/), как они используют GPU и проведите тест-драйв на любом наборе фичей

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 6.4. High tech. Low life** (0.5 балла)

Если вы следите за новостями, то, наверное, заметили появление хайповейших LLM. Злые языки утверждают, что обычному работяге фит предиктору не место в мире будущего, и его заменит ИИ. Давайте в этом (раз)убедимся.

Попробуйте:
1. Спросить у вашей любимой нейросети, какие признаки она может для вас придумать. Можете опираться на пункты выше, можете придумать что-то свое. Но помните, что как говорится, какой стол, такой и стул, поэтому пишите промпты с умом.
2. Показать, что нейросеть вам посоветовала, и реализовать это
3. Проанализировать результат и сделать решительный вывод, хуже ли вы, чем языковая модель.

Попытайтесь либо вспомнить, либо посмотреть, что у нас ещё есть в данных. Там достаточно много полезной информации, которую мы либо совсем никак не брали, либо брали, но поверхностно, либо брали, но можно сделать ещё круче, старые пункты тоже можно доработать

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 6.5. Отбор признаков** (0.5 балла)

Когда признаков становится так много, что ваша оперативка начинает рыдать, а модель переобучается, как чёрт, поможет только одно средство — отбор фичей!
Это первое и единственное задание, в котором <font color="#cb9255">**выбора**</font> аж три:

<table width="100%" border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th width="33%">
      <font color="#cb9255">Sequential Feature Selector</font>
    </th>
    <th width="33%">
      <font color="#cb9255">Greedy Selection</font>
    </th>
    <th width="33%">
      <font color="#cb9255">Recursive Feature Elimination</font>
    </th>
  </tr>
  <tr>
    <td valign="top">
      Делаем итеративно. На каждой итерации <br>
      оцениваем важность фичей по их <br>
      импортансу (<code>coef_</code>), берём <br>
      топ‑n худших, выкидываем, go to 0.
    </td>
    <td valign="top">
      Перебираем все комбинации признаков <br>
      и выбираем наилучшую. Звучит тупо, <br>
      но комбинации можно брать по группам <br>
      (например, тексты, агрегации средних <br>
      и т.д.), тогда это не так долго <br>
      <b>(2 часа на 200 признаков)</b>.
    </td>
    <td valign="top">
      Идём с конца и выкидываем по признаку. <br>
      На каждом шаге обучаем по одной модели <br>
      без одного признака (обучаем d‑1 моделей), <br>
      выбираем из них худшую — такой признак <br>
      и устраняем.
    </td>
  </tr>
  <tr>
    <td valign="top">
      Быстро <b>(около 20 минут <br>
      на 200 фичах)</b>, но веса линейной <br>
      регрессии плохо оценивают важность <br>
      фичей; это лучше работает для <br>
      сильных моделей.
    </td>
    <td valign="top">
      Не теряем интеракции. Баланс <br>
      скорость–качество.
    </td>
    <td valign="top">
      Возмутительно долго <b>(10 часов <br>
      на 200 признаков)</b>, но гарантирует <br>
      минимальные потери в качестве.
    </td>
  </tr>
</table>


<div style="border-left: 5px solid #647cb8; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Это улучшит качество, если вы уже страдаете от миллиарда малополезных фичей. Но для получения балла это не нужно, только верный алгоритм

</div>

Сделайте что-нибудь из этого и проанализируйте эффект. Не стесняйтесь модифицировать схему — удалять по несколько фичей за шаг, параллелить и так далее, пункт времязатратный

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

### Заключение и оценивание

Каждая из задач в ноутбуке имеет свою стоимость (указана в скобках рядом с задачей). При этом важно уточнить разницу между баллами за ноутбук и дополнительными баллами за позицию на приватном лидерборде в соревновании на Kaggle:

1. **Максимум за код/ноутбуки — 8.0 баллов.**
   То есть, независимо от суммарной теоретической суммы всех подпунктов в тексте задания, за реализацию в ноутбуке можно получить не более 8 баллов (6 за базу и 2 за продвинутый).

2. **Максимально возможная оценка за всю работу — 13.0 баллов.**
   Остальные до 5.0 баллов начисляются за результаты в соревновании на Kaggle (лидерборд), при выполненном и загруженном в систему Anytask ноутбуке.

Баллы за сореву состоят из трёх частей: трешхолды (до 2 баллов), процентильный бонус (до 2.0 баллов) и бонус за попадание в топ-10 (до 1.0 балла). Суммарный вклад соревнования не может превышать 5.0 баллов.

**A. Процентильный балл (не суммируется):**
* Если вы пробили трешхолд-9 (качество 0.34) — +1 балл.
* Если вы пробили трешхолд-10 (качество 0.36) — +2 балла.

**Б. Процентильный балл:**

* Если вы только прошли трешхолд-10, то баллов вы не получите.
* Если вы обогнали ≥ 10% участников, побивших трешхолд-10 — +0.5 балла.
* Если вы обогнали ≥ 30% участников — +1.0 балла.
* Если вы обогнали ≥ 60% участников — +1.5 балла.
* Если вы обогнали ≥ 90% участников (т.е. попали в топ 10%) — +2.0 балла.

**В. Балл за попадание в топ-10:**

* 1-е место — +1.00 балла
* 2-е–3-е место — +0.75 балла
* 4-е–6-е место — +0.50 балла
* 7-е–10-е место — +0.25 балла

Пример расчёта

* Вы сделали ноутбуки и получили за них 7.0 / 8.0.
* Вы, тем не менее, побили трешхолд-10 → +2.0
* На лидерборде вы, зайка, обогнали 10% участников, побивших трешхолд 10 → процентильный бонус +2.0.
* Ваша позиция — 3-е место → топ-10 бонус +0.75.
* Итого: 7.0 + 2.0 + 2.0 + 0.75 = 11.75.

Можете свериться с картинкой (левая граница не включительно)

<img src="https://i.postimg.cc/nhb25b42/newplot.png" height=720 width=1280>

**Требование к воспроизводимости**

Баллы за соревнование начисляются **только** при наличии пайплайна или ноутбука, который подтверждает результат лучшего сабмита. Такое решение нужно сдавать вместе с базовым и продвинутым ноутбуками и своим ников в kaggle в Anytask ассистенту. Он должен выполнять всё автоматически при запуске ноутбука: при последовательном исполнении всех ячеек ноутбука (без ручных вмешательств) он должен воспроизвести предобработку, обучение/инференс и сгенерировать итоговый CSV-файл с прогнозами, используемый для сабмита.